# **Lecture:** CPU vs GPU for computer vision

![](intelvsnvidia.png)

### Learning Objectives

- Explain the conceptual difference between CPUs and GPUs
- Understand what a *core* is and why it matters
- Describe why GPUs are essential in computer vision
- Run simple GPU‑accelerated operations on your laptop's GPU (if your laptop doesn’t have an integrated GPU, you’ll be provided access to an NVIDIA Jetson Nano 2GB Developer Kit via its IP address, so you can connect using `ssh`)
- Compare CPU vs GPU performance on matrix operations

### Motivation: Why Do We Need GPUs in Computer Vision?

Modern computer vision relies heavily on:
- Convolutions  
- Matrix multiplications  
- Pixel‑wise operations  
- Deep neural networks  

These tasks involve **millions of independent, repetitive operations**.

A CPU can do them, but slowly (designed for data-level parallelism (DLP))

A GPU can do **thousands of them at the same time** (designed for instruction-level parallelism (ILP) and complex branching)

**Analogy:**  
- CPU → 4 senior developers solving different complex bugs sequentially
- GPU → 500 junior developers running the same function on different inputs in parallel

### CPU vs GPU: Architecture and Purpose

##### CPU (Central Processing Unit)
- 2–16 cores (typical consumer laptops and PCs)  
- Each core is complex and fast  
- Excellent for sequential tasks  
- Ideal for logic, control flow, operating systems  

##### GPU (Graphics Processing Unit)
- Hundreds or thousands of cores  
- Each core is simple  
- Designed for parallel workloads  
- Ideal for linear algebra, images, deep learning  

##### Comparison Table

\begin{array}{lcc}
\hline
\textbf{Aspect} & \textbf{CPU (ILP / TLP)} & \textbf{GPU (DLP)} \\
\hline
\text{Parallelism} & \text{Instruction / Thread Level} & \text{Data Level} \\
\text{Execution Model} & \text{Few complex threads} & \text{Massive lightweight threads} \\
\text{Scalability} & \text{Limited} & \text{High} \\
\text{Memory Bandwidth} & \text{Moderate} & \text{High} \\
\text{Control Flow} & \text{Efficient} & \text{Divergence penalty} \\
\text{Limitation} & \text{Low parallel scaling} & \text{Control complexity} \\
\hline
\end{array}

### What Is a Core?

A **core** is a processing unit capable of executing instructions.

- CPU core → powerful, versatile, handles complex logic  
- GPU core → simple, lightweight, but massively replicated  

On the **Jetson Nano 2GB**:
- CPU: 4 ARM A57 cores  
- GPU: 128 CUDA cores  

### Conceptual Example: Matrix Multiplication

Given:

$$
A = \begin{bmatrix}
1 & 2 \\
3 & 4
\end{bmatrix},
\quad
B = \begin{bmatrix}
5 & 6 \\
7 & 8
\end{bmatrix}
$$

Each element of the result matrix \(C\) is computed **independently**.

#### CPU
- Computes one element at a time (or up to 4 in parallel)

#### GPU
- Can compute **128 elements at once** on the Jetson Nano  
- For large matrices (e.g., 2000×2000), the speed difference is dramatic  

This is exactly what happens inside neural networks.

### Practical Example on Jetson Nano (PyTorch)

#### Check if the GPU is available

In [4]:
import torch
torch.cuda.is_available()

True

In [5]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4050 Laptop GPU


#### Matrix multiplication on the GPU

In [6]:
import torch
device = torch.device("cuda")

A = torch.randn(2000, 2000, device=device)
B = torch.randn(2000, 2000, device=device)

C = A @ B

#### CPU vs GPU timing

In [7]:
import time

# CPU
A_cpu = torch.randn(2000, 2000)
B_cpu = torch.randn(2000, 2000)

start = time.time()
C_cpu = A_cpu @ B_cpu
print("CPU:", time.time() - start)

# GPU
A_gpu = A_cpu.to("cuda")
B_gpu = B_cpu.to("cuda")

torch.cuda.synchronize()
start = time.time()
C_gpu = A_gpu @ B_gpu
torch.cuda.synchronize()
print("GPU:", time.time() - start)

CPU: 0.07307004928588867
GPU: 0.008106708526611328


### Didactic Example: Vector Addition in CUDA

This shows how a GPU kernel works.

#### CUDA kernel (conceptual)


```cuda
__global__
void vector_add(float* a, float* b, float* c) {
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}
```

Each thread computes one element of the output vector.

<div style="text-align: center;">
  <img src="cpuvsgpu.jpeg" style="width: 40%;">
</div>

### Why This Matters in Computer Vision

Most CV operations are parallel by nature:
- Convolutions → each output pixel is independent  
- Filters → parallel  
- Normalization → parallel  
- Neural networks → huge matrix multiplications  

GPUs turn minutes into milliseconds.

### Using the Jetson Nano GPU in CV

### GPU‑accelerated libraries:
- PyTorch (CUDA)  
- TensorFlow (GPU)  
- OpenCV with CUDA  
- cuDNN  
- TensorRT  
- Jetson Inference  

### **Step 1:** Imports

In [8]:
# Imports
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import time
import pandas as pd
import os
import platform
import subprocess

### **Step 2:** Select device (CPU or GPU)

In [9]:
FORCE_CPU = False  # change to True to use CPU

In [10]:
def get_device():
    if FORCE_CPU:
        return "cpu"
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

device = get_device()
print("Using device:", device)

Using device: cuda


In [11]:
cpu_name = subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"]).decode().strip()
cpu_name

FileNotFoundError: [WinError 2] El sistema no puede encontrar el archivo especificado

In [12]:
# =========================
# CPU INFO (CROSS-PLATFORM)
# =========================

def get_cpu_name():
    try:
        system = platform.system()

        if system == "Windows":
            return platform.processor()

        elif system == "Darwin":
            return subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"]
            ).decode().strip()

        elif system == "Linux":
            with open("/proc/cpuinfo") as f:
                for line in f:
                    if "model name" in line:
                        return line.split(":")[1].strip()

        return platform.processor() or platform.machine()

    except Exception:
        return "Unknown CPU"


cpu_name = get_cpu_name()
cpu_cores = os.cpu_count()


# =========================
# GPU INFO (BASED ON DEVICE)
# =========================

def get_gpu_info(device):
    gpu_name = "No dedicated GPU detected"
    gpu_cores = "N/A"

    try:
        if device == "cuda" and torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0)
            props = torch.cuda.get_device_properties(0)
            gpu_cores = props.multi_processor_count

        elif device == "mps":
            gpu_name = "Apple Silicon GPU (MPS)"

    except Exception:
        pass

    return gpu_name, gpu_cores


gpu_name, gpu_cores = get_gpu_info(device)


# =========================
# OUTPUT
# =========================

print("OS:", platform.system())
print("CPU:", cpu_name)
print("CPU cores:", cpu_cores)
print("GPU:", gpu_name)
print("GPU cores:", gpu_cores)

OS: Windows
CPU: Intel64 Family 6 Model 154 Stepping 3, GenuineIntel
CPU cores: 20
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU cores: 20


### **Step 3:** Define a fixed model to test and move it to the designated device

In [13]:
# Model setup (ResNet18 adapted to CIFAR-10)
model = models.resnet18() # default: weights=None
model.fc = torch.nn.Linear(model.fc.in_features, 10)
model = model.to(device)

### **Step 4:** Random data benchmark (inference)

In [14]:
x = torch.randn(32, 3, 224, 224).to(device)

def sync(device):
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()

# Warm-up
for _ in range(10):
    _ = model(x)

start = time.time()
for _ in range(50):
    _ = model(x)
sync(device)
end = time.time()

latency = (end - start) / 50
throughput = 50 / (end - start)

print("Latency:", latency)
print("Throughput:", throughput)

Latency: 0.05613845348358154
Throughput: 17.81310203517565


### **Step 5:** CIFAR-10 dataset train test (time and accuracy)

In [15]:
transform = transforms.ToTensor()

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=32, shuffle=True
)

Files already downloaded and verified


In [16]:
model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

correct = 0
total = 0

start = time.time()

for i, (images, labels) in enumerate(trainloader):
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

    if i > 20:
        break

sync(device)
end = time.time()

training_time = end - start
accuracy = 100 * correct / total

print("Training time:", training_time)
print("Accuracy:", accuracy)


Training time: 0.8911817073822021
Accuracy: 19.460227272727273


### **Step 6:** Results table and `.csv` export

In [17]:
device_type = "cpu" if device == "cpu" else "gpu"
hardware_detail = f"CPU: {cpu_name} | GPU: {gpu_name}"

results = {
    "device": [device_type],
    "hardware_detail": [hardware_detail],
    "cpu_cores": [cpu_cores],
    "gpu_name": [gpu_name],
    "gpu_cores": [gpu_cores],
    "latency": [latency],
    "throughput": [throughput],
    "training_time": [training_time],
    "accuracy": [accuracy]
}

df = pd.DataFrame(results)

csv_file = "benchmark_results (1).csv"

if os.path.exists(csv_file):
    df_existing = pd.read_csv(csv_file)
    df = pd.concat([df_existing, df], ignore_index=True)

# Normalize device labels to only cpu/gpu.
df["device"] = df["device"].replace({
    "CPU": "cpu",
    "cpu": "cpu",
    "cuda": "gpu",
    "mps": "gpu",
    "Metals Performance Shaders (MPS) for Apple Silicon devices": "gpu"
})

df.to_csv(csv_file, index=False)
df

,device,hardware_detail,cpu_cores,gpu_name,gpu_cores,latency,throughput,training_time,accuracy
0,gpu,"CPU: Intel64 Family 6 Model 154 Stepping 3, Ge...",20,NVIDIA GeForce RTX 4050 Laptop GPU,20.0,0.027716,36.079954,0.379976,15.625000
1,cpu,"CPU: Intel64 Family 6 Model 154 Stepping 3, Ge...",20,No dedicated GPU detected,NaN,0.410711,2.434800,6.515040,15.909091
2,gpu,CPU: Intel(R) Xeon(R) CPU @ 2.00GHz | GPU: Tes...,2,Tesla T4,40.0,0.031540,31.705283,0.890852,17.613636
3,cpu,CPU: AMD EPYC 7B13 | GPU: No dedicated GPU det...,24,No dedicated GPU detected,NaN,0.726155,1.377116,7.075864,17.045455
4,gpu,"CPU: Intel64 Family 6 Model 154 Stepping 3, Ge...",20,NVIDIA GeForce RTX 4050 Laptop GPU,20.0,0.056138,17.813102,0.891182,19.460227


# **Individual challenge**

### **Code**

Students without local GPUs will be provided with a Docker container into the **Jetson Nano**.

Complete the following table with your findings. The goal is to identify the "Sweet Spot" between cost and performance.

| Environment | Device Type | Time (100 Batches) | Speedup (vs Local CPU) | Export Success (ONNX/TRT) |
|-------------|-------------|--------------------|------------------------|---------------------------|
| Local Laptop| CPU         |                    | 1.0x                   |                           |
| Local Laptop| GPU or Jetson Nano (CUDA)  |                    |                        |                           |
| Google Colab| Tesla T4    |                    |                        |                           |
| Google Colab| TPU         |                    |                        |                           |
| Lab PC      | CPU    |                    |                        |                           |

**Activity:** Post your "Speedup" result in the class [shared spreadsheet](https://yachaytecheduec-my.sharepoint.com/:x:/g/personal/mmorocho_yachaytech_edu_ec/IQCcOxYR8qjTRKkClRTv3isMAUEnuqvpj0VH9mVkDXoVbPE?e=w83G0t). Let's find the group average!



### **Questions:** 

1. **To what extent can instruction-level parallelism (ILP) and multithreading in CPUs approximate the data-parallel execution model of GPUs? Where does this approach break down?**

* Instruction-level parallelism (ILP) and multithreading in CPUs allow CPUs to approximate the GPU model by executing multiple operations simultaneously and hiding memory latencies through rapid thread switching, also utilizing SIMD units (such as the AVX-512) to process data vectors in a single cycle. However, this approach fails with massively parallel tasks because the CPU architecture prioritizes latency reduction through complex control logic and large caches, sacrificing the space that a GPU dedicates almost exclusively to ALUs (arithmetic units). This structural limitation, coupled with significantly lower memory bandwidth compared to VRAM, means that the CPU cannot scale performance with massive data volumes, where the energy cost of discovering dynamic parallelism cannot compete with the deterministic and massive execution of a GPU.

2. **If you had to deploy a model for real-time traffic monitoring on a remote highway with no internet, which of the tested hardwares would you choose and why?**

* For this scenario, the best choice would be the GPU (RTX 4050), as it's the only high-performance hardware in your tests that allows for local execution (edge ​​computing) without relying on the connectivity required by the Tesla T4 or the TPU in the cloud. With a 17.14x speedup and the lowest latency ($0.027), this GPU guarantees video processing at the necessary FPS to capture high-speed vehicles in real time, overcoming the limitations of the local CPU and the inefficiency of the lab's CPU. Furthermore, by processing data on-site, energy consumption is minimized compared to less optimized systems, allowing the device to operate autonomously using batteries or solar panels at the remote location.

3. **Who achieved the top 5 throughput values in the class?**

    1. Mateo Pilaquinga - Colab TPU - 478,7242
    2. Luis Cedeño - GPU: NVIDIA GeForce RTX 4070 Laptop GPU - 57,93049
    3. Daniel Rivas - NVIDIA GeForce RTX 3060 Laptop GPU - 50,7179
    4. Jamil Andi - NVIDIA GeForce RTX 3060 Laptop GPU - 47,326
    5. Marck Chiza - NVIDIA GeForce RTX 4050 Laptop GPU - 36,079954
    

### **Grading Rubric** 

| Criterion | Points (out of 10) |
|---|---:|
| 1. Environment and Hardware Discovery | 3.0 |
| 2. Comparative Performance Analysis | 3.5 |
| 3. Engineering Reasoning (Section 8) | 3.5 |
| **Total** | **10.0** |  



| Criterion | Excellent | Proficient | Developing | Beginning |
|---|---|---|---|---|
| 1. Environment and Hardware Discovery (3.0) | Correctly identifies OS, CPU, and accelerator; reports key specs clearly. | Identifies most hardware info with minor omissions. | Basic/incomplete hardware report; some inaccuracies. | Missing or mostly incorrect hardware identification. |
| 2. Comparative Performance Analysis (3.5) | Computes speedup correctly, compares devices meaningfully, and explains latency/throughput tradeoffs. | Comparison is correct but interpretation lacks depth. | Basic comparison with weak or partially incorrect analysis. | No meaningful comparison or incorrect conclusions. |
| 3. Engineering Reasoning (Section 8) (3.5) | Answers are technically strong, well-justified, and tied to architecture/deployment context. | Answers are mostly correct but somewhat superficial. | Partial understanding with important conceptual gaps. | Missing answers or mostly incorrect reasoning. |

---

<p style="text-align: right; font-size:14px; color:gray;">
<b>Prepared by:</b><br>
Manuel Eugenio Morocho-Cayamcela
</p>